# 02 – Transaction Monitoring Rules
Rules are written in `/sql` and executed here. Labels (`Is_laundering`, `Laundering_type`) are **never** used inside rules, only for evaluation in notebook 03.

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect("../aml.duckdb")
con.sql("SET enable_progress_bar = false")

def run_sql(path):
    """Read a .sql file and run it against the database."""
    with open(path) as f:
        con.sql(f.read())
    print(f"Ran {path}")

In [5]:
con.sql("""
SELECT COUNT(*) AS n,
       SUM(CASE WHEN Sender_account = Receiver_account THEN 1 ELSE 0 END) AS same_account,
       COUNT(DISTINCT Sender_account)   AS n_senders,
       COUNT(DISTINCT Receiver_account) AS n_receivers
FROM tx
WHERE Payment_type = 'Cash Deposit'
""").df()

,n,same_account,n_senders,n_receivers
0,225206,0.0,24328,15179


In [6]:
con.sql("""
WITH weekly AS (
    SELECT Receiver_account          AS account,
           date_trunc('week', ts)    AS wk,
           COUNT(*)                  AS n_deposits,
           SUM(Amount)               AS total_deposited
    FROM tx
    WHERE Payment_type = 'Cash Deposit'
    GROUP BY account, wk
)
SELECT quantile_cont(n_deposits, 0.50)      AS p50_count,
       quantile_cont(n_deposits, 0.99)      AS p99_count,
       MAX(n_deposits)                      AS max_count,
       quantile_cont(total_deposited, 0.99) AS p99_weekly_total
FROM weekly
""").df()

,p50_count,p99_count,max_count,p99_weekly_total
0,1.0,3.0,9,9282.1196


## R1 – Structuring / smurfing

In [ ]:
run_sql("../sql/r1_cash_smurfing.sql")

con.sql("""
SELECT COUNT(*)                         AS alert_weeks,
       COUNT(DISTINCT account)          AS accounts_alerted,
       ROUND(AVG(n_deposits), 1)        AS avg_deposits,
       ROUND(AVG(n_depositors), 1)      AS avg_depositors,
       ROUND(AVG(total_deposited), 0)   AS avg_weekly_total
FROM r1_alerts
""").df()

Ran ../sql/r1_structuring.sql


,alert_weeks,accounts_alerted,avg_deposits,avg_depositors,avg_weekly_total
0,1209,979,3.5,2.1,11308.0


In [11]:
con.sql("""
SELECT * FROM r1_alerts
ORDER BY total_deposited DESC
LIMIT 10
""").df()

,account,wk,n_deposits,n_depositors,total_deposited,max_deposit,rule
0,2264961913,2022-12-05,7,1,26869.40,6728.76,R1_structuring
1,4345836865,2023-06-12,9,2,24196.81,5180.19,R1_structuring
2,8020264563,2023-05-15,7,2,23557.25,5771.28,R1_structuring
3,2817594182,2023-08-14,8,2,22310.14,5406.53,R1_structuring
4,3062851970,2023-05-22,5,2,21420.57,5279.18,R1_structuring
5,927895922,2022-10-10,6,2,21283.83,5114.26,R1_structuring
6,2717787457,2022-12-26,5,3,21067.78,5888.71,R1_structuring
7,6436958188,2022-10-31,5,3,20667.20,5851.24,R1_structuring
8,2788686026,2023-07-31,5,2,20336.43,5933.81,R1_structuring
9,2319300432,2023-08-14,6,1,20030.75,3854.00,R1_structuring


Observation: R1 generated 1,209 alert-weeks across 979 accounts (~26 per week). Top alerts show 5–9 sub-threshold deposits totalling 20,000–27,000 in a single week, often from multiple depositors, consistent with structuring and smurfing.

## R2 – Fan-in

In [12]:
run_sql("../sql/r2_fan_in.sql")

con.sql("""
SELECT COUNT(*)                        AS alert_weeks,
       COUNT(DISTINCT account)         AS accounts_alerted,
       ROUND(AVG(n_counterparties), 1) AS avg_senders,
       ROUND(AVG(total_amount), 0)     AS avg_weekly_total
FROM r2_alerts
""").df()

Ran ../sql/r2_fan_in.sql


,alert_weeks,accounts_alerted,avg_senders,avg_weekly_total
0,13067,10981,14.4,1566633.0


Observation: R2 generated 13,067 alert-weeks across 10,981 accounts (~284 per week). It shows the average of 14 senders. Overall, its a heavy queue many of which are probably legitimate businesses or "Normal_Fan_In" transactions.

## R3 – Fan-out (threshold research)

In [13]:
con.sql("""
WITH weekly AS (
    SELECT Sender_account                    AS account,
           date_trunc('week', ts)            AS wk,
           COUNT(DISTINCT Receiver_account)  AS n_receivers
    FROM tx
    GROUP BY account, wk
)
SELECT quantile_cont(n_receivers, 0.50)  AS p50,
       quantile_cont(n_receivers, 0.99)  AS p99,
       quantile_cont(n_receivers, 0.999) AS p999,
       MAX(n_receivers)                  AS max_receivers
FROM weekly
""").df()

,p50,p99,p999,max_receivers
0,1.0,12.0,23.0,33


## R3 – Fan-out

In [14]:
run_sql("../sql/r3_fan_out.sql")

con.sql("""
SELECT COUNT(*)                        AS alert_weeks,
       COUNT(DISTINCT account)         AS accounts_alerted,
       ROUND(AVG(n_counterparties), 1) AS avg_receivers,
       ROUND(AVG(total_amount), 0)     AS avg_weekly_total
FROM r3_alerts
""").df()

Ran ../sql/r3_fan_out.sql


,alert_weeks,accounts_alerted,avg_receivers,avg_weekly_total
0,7464,7459,20.9,2156721.0


Observation: R3 generated 7,464 alert-weeks across 7,459 accounts (~162 per week). It shows one almost one alert per account which means it happen once rather than repeat behaviour

In [15]:
con.sql("""
SELECT 'R2' AS rule, ROUND(MEDIAN(total_amount), 0) AS median_weekly_total FROM r2_alerts
UNION ALL
SELECT 'R3', ROUND(MEDIAN(total_amount), 0) FROM r3_alerts
""").df()

,rule,median_weekly_total
0,R2,1712150.0
1,R3,2167581.0


## R4 – Pass-through

In [16]:
run_sql("../sql/r4_pass_through.sql")
con.sql("SELECT COUNT(*) AS n_pairs, COUNT(DISTINCT account) AS n_accounts FROM r4_pairs").df()

Ran ../sql/r4_pass_through.sql


,n_pairs,n_accounts
0,834737,30960


In [17]:
con.sql("""
WITH weekly AS (
    SELECT account,
           date_trunc('week', in_ts) AS wk,
           COUNT(*)                  AS n_pairs
    FROM r4_pairs
    GROUP BY account, wk
)
SELECT COUNT(*)                       AS account_weeks_with_pairs,
       quantile_cont(n_pairs, 0.50)   AS p50,
       quantile_cont(n_pairs, 0.90)   AS p90,
       quantile_cont(n_pairs, 0.99)   AS p99,
       MAX(n_pairs)                   AS max_pairs
FROM weekly
""").df()

,account_weeks_with_pairs,p50,p90,p99,max_pairs
0,86481,1.0,8.0,72.0,13708


In [18]:
con.sql("""
WITH weekly AS (
    SELECT account,
           date_trunc('week', in_ts) AS wk,
           COUNT(DISTINCT CAST(from_account AS VARCHAR) || '_' || CAST(in_ts AS VARCHAR)) AS n_passed,
           COUNT(DISTINCT to_account) AS n_destinations
    FROM r4_pairs
    GROUP BY account, wk
)
SELECT quantile_cont(n_passed, 0.50) AS p50,
       quantile_cont(n_passed, 0.90) AS p90,
       quantile_cont(n_passed, 0.99) AS p99,
       MAX(n_passed)                 AS max_passed
FROM weekly
""").df()

,p50,p90,p99,max_passed
0,1.0,6.0,60.0,229


In [19]:
run_sql("../sql/r4_pass_through.sql")

con.sql("""
SELECT COUNT(*)                      AS alert_weeks,
       COUNT(DISTINCT account)       AS accounts_alerted,
       ROUND(MEDIAN(n_passed), 1)    AS median_passed,
       ROUND(MEDIAN(n_sources), 1)   AS median_sources,
       ROUND(MEDIAN(n_destinations), 1) AS median_destinations
FROM r4_alerts
""").df()

Ran ../sql/r4_pass_through.sql


,alert_weeks,accounts_alerted,median_passed,median_sources,median_destinations
0,1571,1571,64.0,7.0,2.0


Observation: R4 generated 1,571 alert-weeks across 1,571 accounts (~34 per week) with each account alerting once, short and intense. The distinct counting reduced the max value to 229 fixing the inflation. There many in, few out money from about 7 sources funnelled onward to about 2 accounts which is the signature of a mule collection account consolidating funds before passing them up the chain.

## R5 – High-risk geography

In [20]:
run_sql("../sql/r5_high_risk_geo.sql")

con.sql("""
SELECT hr_country, COUNT(*) AS n_tx, COUNT(DISTINCT account) AS n_accounts
FROM r5_tx
GROUP BY hr_country
ORDER BY n_tx DESC
""").df()

Ran ../sql/r5_high_risk_geo.sql


,hr_country,n_tx,n_accounts
0,Turkey,56086,5124
1,Albania,55969,5113
2,UAE,55449,5038
3,Nigeria,32121,3598
4,Morocco,24699,2985
5,Pakistan,1811,590


Observation: Turkey shows more transactions here than in Q4 because R5 counts payments in both directions, to and from, while Q4 only counted the receiver side.

## R5 – Fan-out (threshold research)

In [21]:
con.sql("""
WITH weekly AS (
    SELECT account,
           date_trunc('week', ts) AS wk,
           COUNT(*)               AS n_tx,
           SUM(Amount)            AS total_amount
    FROM r5_tx
    GROUP BY account, wk
)
SELECT COUNT(*)                            AS account_weeks,
       quantile_cont(total_amount, 0.50)   AS p50_total,
       quantile_cont(total_amount, 0.90)   AS p90_total,
       quantile_cont(total_amount, 0.99)   AS p99_total,
       quantile_cont(n_tx, 0.99)           AS p99_count
FROM weekly
""").df()

,account_weeks,p50_total,p90_total,p99_total,p99_count
0,73161,5239.38,99100.39,314045.15,14.0


In [29]:
run_sql("../sql/r5_high_risk_geo.sql")

con.sql("""
SELECT COUNT(*)                      AS alert_weeks,
       COUNT(DISTINCT account)       AS accounts_alerted,
       ROUND(MEDIAN(n_tx), 1)        AS median_tx,
       ROUND(MEDIAN(total_amount), 0) AS median_total
FROM r5_alerts
""").df()

Ran ../sql/r5_high_risk_geo.sql


,alert_weeks,accounts_alerted,median_tx,median_total
0,2252,2230,13.0,267595.0


In [30]:
con.sql("""
SELECT countries, COUNT(*) AS alert_weeks
FROM r5_alerts
GROUP BY countries
ORDER BY alert_weeks DESC
LIMIT 10
""").df()

,countries,alert_weeks
0,Turkey,478
1,UAE,448
2,Albania,435
3,Nigeria,272
4,Morocco,217
5,"Albania, UAE",63
6,"Turkey, UAE",63
7,"Albania, Turkey",50
8,"Nigeria, Turkey",35
9,"Albania, Nigeria",32


Observation: R5 generated 2,252 alert-weeks across 2,230 accounts (~49 per week), almost all one-off account weeks. The rows showing two countries (for example "Albania, Turkey") are accounts moving money through multiple listed jurisdictions in the same week. That's a stronger red flag than a single country, because routing funds across several weak AML regimes is a classic layering technique.

In [31]:
con.close()